# Module 3 — FaithEval dose-response + orthogonal-projection ablation

**v2 §3 hypothesis.** Desperation steering increases hallucination on unanswerable-context questions; orthogonally projecting the desperation direction out of the residual stream decreases hallucination relative to baseline.

**Prerequisite:** M1 must have completed (vectors at `outputs/m1_vectors/desperation.npy`). M2 is helpful for context (max usable α) but M3 sweeps the full α grid regardless to characterize the dose-response curve.

**Three arms:**
1. **Baseline** — Gemma-2-9B-IT on FaithEval-unanswerable, no hook. Reference rates.
2. **Steered** — desperation hook at α ∈ {0.025, 0.05, 0.075, 0.1}. Expect hallucination rate to *increase* monotonically with α.
3. **Ablated** — orthogonal-projection hook (`h' = h − (h·v̂)v̂`). Expect hallucination rate to *decrease* relative to baseline.

**Pass criteria:**
- Monotonic dose-response in arm 2 (each higher α has higher hallucination than the lower).
- Gap between arm 1 (baseline) and arm 3 (ablated) in the predicted direction (ablated ≤ baseline).

**Cost:** 2,492 FaithEval prompts × 6 runs (baseline + 4 α + ablated) = ~15k generations on 9B. At ~2.5s/prompt greedy decode, ~10 hr A100 wall. Plus ~$2 in Haiku classifier calls. Total ~$15.

**Outputs:** `outputs/m3/faitheval_{baseline,steered_a*,ablated}.csv`, `outputs/m3/dose_response.csv`, `outputs/m3/decision.txt`.

**Smoke-test before the full run:** edit `LIMIT = None` in Cell 2 to `LIMIT = 50` for a 6-cell × 50-prompt smoke (~10 min total) that validates the full pipeline end-to-end. Set back to `None` for the real run.

## Cell 1 — env setup

In [ ]:
from google.colab import userdata
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
!git clone https://github.com/BraydenFeng/Algoverse.git || (cd Algoverse && git pull)
%cd Algoverse

!pip install -q -r requirements.txt

## Cell 2 — load model, desperation vector, calibrate ||h||

In [ ]:
import sys
sys.path.insert(0, '.')

from pathlib import Path
import numpy as np

from src.lib.config import load_config
from src.lib.model_load import load_gemma
from src.steering import load_emotion_vector, estimate_residual_norm

# Toggle for smoke test vs full run. None = all 2,492 FaithEval prompts per arm.
LIMIT = None

cfg = load_config()
layer = cfg['models']['primary']['extraction_layer']
alphas = cfg['steering']['alpha_sweep']

# desperation only — the other emotion vectors are M1 controls, not M3 treatments
desperation = load_emotion_vector('desperation')
print(f'desperation vector: shape={desperation.shape}, ||v||={np.linalg.norm(desperation):.4f}')

model, tokenizer = load_gemma(variant='primary')
print(f'loaded {model.config._name_or_path}, n_layers={model.config.num_hidden_layers}, steering layer={layer}')

# calibrate residual norm on the neutral corpus, same way M2 did. If M2 ran earlier
# this could be cached, but recomputing keeps M3 standalone-runnable.
data_dir = Path(cfg['paths']['data_dir'])
neutral_texts = [p.read_text(encoding='utf-8') for p in sorted((data_dir / 'stories' / 'neutral').glob('*.txt'))]
norm_scale = estimate_residual_norm(
    model, tokenizer, layer=layer,
    calibration_texts=neutral_texts,
    token_skip=cfg['extraction']['token_skip'],
)
print(f'mean ||h||@L{layer} = {norm_scale:.2f}')
print(f'α-sweep: {alphas}')
print(f'LIMIT = {LIMIT} (None means all 2,492 prompts per arm)')

## Cell 3 — baseline arm (no hook)

~1.5 hr on A100 at full LIMIT=None. Checkpoint CSV saves every 100 prompts so a crash mid-arm resumes cleanly via the same path.

In [ ]:
from src.faitheval_eval import run_eval, summary

outputs_dir = Path(cfg['paths']['outputs_dir']) / 'm3'
outputs_dir.mkdir(parents=True, exist_ok=True)

df_baseline = run_eval(
    model, tokenizer,
    limit=LIMIT,
    pre_forward_hook=None,
    checkpoint_path=outputs_dir / 'faitheval_baseline.csv',
    checkpoint_every=100,
)
baseline_summary = summary(df_baseline)
print(baseline_summary)

## Cell 4 — steered arm (dose-response sweep)

Four runs at α ∈ {0.025, 0.05, 0.075, 0.1}, each with the desperation steering hook. ~6 hr total at full LIMIT=None. Each α's CSV is separate so partial completion is durable.

In [ ]:
from src.steering import make_steering_hook_factory

steered_summaries = {}
for alpha in alphas:
    print(f'\n=== steered α={alpha} ===')
    hook_factory = make_steering_hook_factory(
        vector=desperation, layer=layer, alpha=alpha, norm_scale=norm_scale,
    )
    df = run_eval(
        model, tokenizer,
        limit=LIMIT,
        pre_forward_hook=hook_factory,
        checkpoint_path=outputs_dir / f'faitheval_steered_a{alpha}.csv',
        checkpoint_every=100,
    )
    s = summary(df)
    steered_summaries[alpha] = s
    print(s)

## Cell 5 — ablated arm (project-out)

Orthogonally project the desperation direction OUT of the residual stream at the steering layer on every forward pass. No α — this is a hard projection. ~1.5 hr at full LIMIT=None.

In [ ]:
from src.steering import make_ablation_hook_factory

print('=== ablated (project-out desperation) ===')
ablation_factory = make_ablation_hook_factory(vector=desperation, layer=layer)
df_ablated = run_eval(
    model, tokenizer,
    limit=LIMIT,
    pre_forward_hook=ablation_factory,
    checkpoint_path=outputs_dir / 'faitheval_ablated.csv',
    checkpoint_every=100,
)
ablated_summary = summary(df_ablated)
print(ablated_summary)

## Cell 6 — aggregate, dose-response, decision

Build the dose-response table (α → hallucination rate, refusal rate), check monotonicity, check the baseline-vs-ablated gap, write the decision file.

In [ ]:
import pandas as pd

rows = []
rows.append({'arm': 'baseline', 'alpha': 0.0, **baseline_summary})
for a in alphas:
    rows.append({'arm': 'steered', 'alpha': a, **steered_summaries[a]})
rows.append({'arm': 'ablated', 'alpha': None, **ablated_summary})

dose = pd.DataFrame(rows)
dose.to_csv(outputs_dir / 'dose_response.csv', index=False)
print(dose.to_string(index=False))

# monotonicity check on the steered arm: hallucination rate should rise with α
steered = dose[dose['arm'] == 'steered'].sort_values('alpha')
halluc_series = steered['hallucination_rate'].tolist()
monotonic = all(halluc_series[i] <= halluc_series[i+1] for i in range(len(halluc_series)-1))

# baseline vs ablated gap (predicted: ablated ≤ baseline)
baseline_h = baseline_summary['hallucination_rate']
ablated_h = ablated_summary['hallucination_rate']
ablation_gap = baseline_h - ablated_h  # positive means ablation reduced hallucination as predicted

lines = ['Module 3 — FaithEval dose-response + ablation', '=' * 50, '']
lines.append(f'n_prompts per arm: {baseline_summary["n"]}')
lines.append(f'norm scale ||h||@L{layer}: {norm_scale:.2f}')
lines.append('')
lines.append('hallucination rates:')
lines.append(f'  baseline  (α=0)    : {baseline_h:.4f}')
for a in alphas:
    lines.append(f'  steered   α={a:.3f}: {steered_summaries[a]["hallucination_rate"]:.4f}')
lines.append(f'  ablated   (proj-out): {ablated_h:.4f}')
lines.append('')
lines.append(f'monotonic dose-response: {monotonic}')
lines.append(f'ablation gap (baseline − ablated): {ablation_gap:+.4f} (positive = ablation reduced hallucination)')

decision = '\n'.join(lines)
print('\n' + decision)
(outputs_dir / 'decision.txt').write_text(decision, encoding='utf-8')

## Outputs

- `outputs/m3/faitheval_baseline.csv` — per-prompt results for the no-hook arm
- `outputs/m3/faitheval_steered_a{α}.csv` — per-prompt results for each steered α
- `outputs/m3/faitheval_ablated.csv` — per-prompt results for the project-out arm
- `outputs/m3/dose_response.csv` — aggregated rate-per-arm table
- `outputs/m3/decision.txt` — monotonicity + ablation-gap summary

## Interpretation (human-owned)

The numbers in `dose_response.csv` are computed, but the result claim isn't. Things to think about before writing this up:

- **Monotonicity at small effect sizes.** `monotonic=True` on noisy 2,492-prompt rates is not the same as a meaningful dose-response. Bootstrap each arm's hallucination rate (sample with replacement over the prompts) and compare CIs.
- **Refusal vs hallucination decomposition.** If steering drives `refuses → off_topic` rather than `refuses → fabricates`, the hallucination rate looks unchanged but the model is still being damaged. Watch the `off_topic_rate` and `by_method` fields in `summary()` for each arm.
- **The M0 prompt template confound.** `config.yaml`'s FaithEval prompt template explicitly says "the answer should be 'unknown'." That nudges the baseline refusal rate up artificially. Steering may push the model past that nudge — which is the predicted effect — or it may just be undoing the nudge. Disentangling these would require running the baseline arm with and without the explicit-refusal instruction, which is outside v2 scope but worth flagging in the limitations section.
- **Capability gate from M2.** If M3 finds a big hallucination delta at α=0.1, but M2 said α=0.1 drops MMLU by 8pts, the steering is doing something but not specifically. The headline α for the paper should be the M2 max-usable α, with the higher α values shown for the curve.

## Next

M4 (Imai causal mediation through the unknown-entity SAE latent). Blocked until `config.yaml`'s `sae.unknown_entity_feature_idx` is filled in from Ferrando v2 (arXiv:2411.14257) Appendix Q.